In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
dataset_list = ["ARC_Challenge", "CommonSenseQA", "MMLU", "OpenBookQA"
                ]
model_list = [
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "Qwen/Qwen1.5-4B",
    "Qwen/Qwen1.5-0.5B",
    "Qwen/Qwen1.5-1.8B",
    "openai-community/gpt2",
    "openai-community/gpt2-large",
    "openai-community/gpt2-medium",
  ]

In [3]:
def get_mean_std(data): 
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)
    return mean, std

In [4]:
def get_data_list(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        results = [json.loads(line) for line in f]

    group_base = ["prompt_1", "prompt_2", "prompt_3"]
    group_fewer = ["prompt_10", "prompt_11", "prompt_12"]
    group_more = ["prompt_13", "prompt_14", "prompt_15"]
    prompt_1_list = np.stack([r["prompt_key0"] for r in results], axis=0)  
    prompt_2_list = np.stack([r["prompt_key1"] for r in results], axis=0) 

    all_grad_list = np.stack([r["grads_norm_list"] for r in results], axis=0)              # shape: (num_samples, dim)
    all_delta_z_list = np.stack([r["delta_z_norm_list"] for r in results], axis=0)
    
    delta_log_prob_norm_list = [r["delta_log_prob_norm"] for r in results]

    fewer_grad_list = []
    fewer_delta_z_list = []
    fewer_delta_log_prob_norm_list = []
    more_grad_list = []
    more_delta_z_list = []
    more_delta_log_prob_norm_list = []

    for p1, p2, grad, delta_z, delta_log_prob in zip(prompt_1_list, prompt_2_list, all_grad_list, all_delta_z_list, delta_log_prob_norm_list):
        if p1 == p2:
            continue
        if p1 in group_base and p2 in group_fewer:
            fewer_grad_list.append(grad)
            fewer_delta_z_list.append(delta_z)
            fewer_delta_log_prob_norm_list.append(delta_log_prob)
        elif p1 in group_base and p2 in group_more:
            more_grad_list.append(grad)
            more_delta_z_list.append(delta_z)
            more_delta_log_prob_norm_list.append(delta_log_prob)

    fewer_grad_list = np.array(fewer_grad_list)
    fewer_delta_z_list = np.array(fewer_delta_z_list)
    more_grad_list = np.array(more_grad_list)
    more_delta_z_list = np.array(more_delta_z_list)
    fewer_upper_bound_list = fewer_grad_list * fewer_delta_z_list
    more_upper_bound_list = more_grad_list * more_delta_z_list

    fewer_grad_mean_list, fewer_grad_std_list = get_mean_std(fewer_grad_list)
    fewer_delta_z_mean_list, fewer_delta_z_std_list = get_mean_std(fewer_delta_z_list)
    more_grad_mean_list, more_grad_std_list = get_mean_std(more_grad_list)
    more_delta_z_mean_list, more_delta_z_std_list = get_mean_std(more_delta_z_list)
    fewer_upper_bound_mean_list, fewer_upper_bound_std_list = get_mean_std(fewer_upper_bound_list)
    more_upper_bound_mean_list, more_upper_bound_std_list = get_mean_std(more_upper_bound_list)

    fewer_result_dict = {
        "grad_mean_list": fewer_grad_mean_list,
        "grad_std_list": fewer_grad_std_list,
        "delta_z_mean_list": fewer_delta_z_mean_list,
        "delta_z_std_list": fewer_delta_z_std_list,
        "upper_bound_mean_list": fewer_upper_bound_mean_list,
        "upper_bound_std_list": fewer_upper_bound_std_list,
        "delta_log_prob_norm_list": fewer_delta_log_prob_norm_list,
    }

    more_result_dict = {
        "grad_mean_list": more_grad_mean_list,
        "grad_std_list": more_grad_std_list,
        "delta_z_mean_list": more_delta_z_mean_list,
        "delta_z_std_list": more_delta_z_std_list,
        "upper_bound_mean_list": more_upper_bound_mean_list,
        "upper_bound_std_list": more_upper_bound_std_list,
        "delta_log_prob_norm_list": more_delta_log_prob_norm_list,
    }
    return fewer_result_dict, more_result_dict

In [5]:
def plot_line(fewer_delta_z_mean_list, 
              fewer_delta_z_std_list,
              more_delta_z_mean_list, 
              more_delta_z_std_list,
              fewer_color, 
              more_color, 
              dataset, 
              model_name_or_path, 
              label):
    plt.figure(figsize=(2.5, 2.5))

    x = np.arange(len(fewer_delta_z_mean_list))
    xticks = [str(i) for i in range(len(x))]
    step = max(1, len(x) // 4)
    
    fewer_mean = np.array(fewer_delta_z_mean_list)
    fewer_std = np.array(fewer_delta_z_std_list)
    fewer_lower = fewer_mean - fewer_std
    fewer_upper = fewer_mean + fewer_std

    more_mean = np.array(more_delta_z_mean_list)
    more_std = np.array(more_delta_z_std_list)
    more_lower = more_mean - more_std
    more_upper = more_mean + more_std

    plt.plot(
        x,
        fewer_delta_z_mean_list,
        label=r"Fewer",
        color=fewer_color,
        linewidth=2
    )
    plt.plot(
        x,
        more_delta_z_mean_list,
        label=r"More",
        color=more_color,
        linewidth=2
    )
    plt.fill_between(
        x,
        fewer_lower,
        fewer_upper,
        color=fewer_color,
        alpha=0.2,
        linewidth=0
    )
    plt.fill_between(
        x,
        more_lower,
        more_upper,
        color=more_color,
        alpha=0.2,
        linewidth=0
    )
    # if line is not None:
    #     plt.axhline(y=line, color="#0095FF", linestyle="-", linewidth=2)

    plt.xticks(x[::step], xticks[::step])

    plt.xlabel("Number of layers")
    plt.title("Fewer vs. More")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    
    save_path = f"../../results/figure_results/how_fewer_more/{model_name_or_path}/{dataset}_{label}.pdf"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()


In [ ]:
def plot_grads_and_deltaz(dataset, model_name_or_path):
    file_path = f"../../results/data_results/misalignment/{model_name_or_path}/{dataset}_result.jsonl"
    fewer_result_dict, more_result_dict = get_data_list(file_path)
    
    fewer_grad_mean_list = fewer_result_dict["grad_mean_list"]
    fewer_grad_std_list = fewer_result_dict["grad_std_list"]
    fewer_delta_z_mean_list = fewer_result_dict["delta_z_mean_list"]
    fewer_delta_z_std_list = fewer_result_dict["delta_z_std_list"]
    fewer_upper_bound_mean_list = fewer_result_dict["upper_bound_mean_list"]
    fewer_upper_bound_std_list = fewer_result_dict["upper_bound_std_list"]
    fewer_delta_log_prob_norm_list = fewer_result_dict["delta_log_prob_norm_list"]

    more_grad_mean_list = more_result_dict["grad_mean_list"]
    more_grad_std_list = more_result_dict["grad_std_list"]
    more_delta_z_mean_list = more_result_dict["delta_z_mean_list"]
    more_delta_z_std_list = more_result_dict["delta_z_std_list"]
    more_upper_bound_mean_list = more_result_dict["upper_bound_mean_list"]
    more_upper_bound_std_list = more_result_dict["upper_bound_std_list"]
    more_delta_log_prob_norm_list = more_result_dict["delta_log_prob_norm_list"]

    fewer_mean_delta_log_prob_norm = np.mean(fewer_delta_log_prob_norm_list)
    more_mean_delta_log_prob_norm = np.mean(more_delta_log_prob_norm_list)

    fewer_color = "#00894b"
    more_color = "#BF1E2E"
    plot_line(fewer_delta_z_mean_list, 
              fewer_delta_z_std_list,
              more_delta_z_mean_list, 
              more_delta_z_std_list,
              fewer_color, 
              more_color, 
              dataset, 
              model_name_or_path, 
              "fewer_more")
   

In [7]:
for dataset in dataset_list:
    for model_name_or_path in model_list:
        plot_grads_and_deltaz(dataset, model_name_or_path)
